# 체이닝 워크플로 — Chaining Workflows

**Skilljar Lesson 03 대응**

이 노트북에서 다루는 내용:
1. 체이닝 헬퍼 함수 (`chain_step`)
2. 3단계 문서 분석 파이프라인
3. Gate 패턴 (중간 품질 검증)
4. 건축공학 적용: 시방서 분석 체이닝

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 체이닝 헬퍼 함수

체이닝의 각 단계를 실행하는 범용 헬퍼 함수입니다.  
이전 단계의 출력이 다음 단계의 입력이 됩니다.

In [ ]:
def chain_step(system_prompt, user_message, model=MODEL):
    """체이닝의 단일 단계를 실행한다.
    
    Args:
        system_prompt: 이 단계의 역할/지시
        user_message: 이전 단계의 출력 또는 초기 입력
        model: 사용할 모델
    
    Returns:
        이 단계의 출력 텍스트
    """
    response = client.messages.create(
        model=model,
        max_tokens=2048,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

print("chain_step 함수 정의 완료")

## §2. 3단계 문서 분석 체이닝

In [ ]:
sample_document = """
Project: High-Rise Office Building (30 floors)
Location: Seoul, South Korea
Structural System: RC core wall + steel moment frame

Key Requirements:
- Seismic Design Category: SDC D (KDS 41 17 00)
- Wind load: Basic wind speed 26 m/s
- Foundation: Mat foundation on rock (bearing capacity 500 kPa)
- Floor system: Composite deck with 150mm RC slab
- Target performance: Immediate Occupancy for design earthquake

Schedule:
- Design phase: 6 months
- Construction: 24 months
- Occupancy: Q4 2028
"""

print("샘플 문서 준비 완료")
print(f"문서 길이: {len(sample_document)} 글자")

In [ ]:
def document_analysis_chain(document_text):
    """3단계 문서 분석 체이닝 파이프라인"""
    
    # Step 1: 핵심 정보 추출
    print("Step 1: 핵심 정보 추출 중...")
    extracted = chain_step(
        system_prompt=(
            "You are a technical document analyst. "
            "Extract the key facts, requirements, and specifications "
            "from the document. Output as a structured list in Korean."
        ),
        user_message=f"Extract key information from:\n\n{document_text}"
    )
    print(f"  -> 추출 완료 ({len(extracted)} 글자)")
    print(f"  -> 미리보기: {extracted[:150]}...\n")
    
    # Step 2: 구조화된 분석 (Step 1의 출력이 입력)
    print("Step 2: 구조화된 분석 중...")
    analysis = chain_step(
        system_prompt=(
            "You are a senior structural engineer. Analyze the following "
            "extracted information and identify: (1) critical requirements, "
            "(2) potential risks, (3) dependencies. Respond in Korean."
        ),
        user_message=f"Analyze these extracted facts:\n\n{extracted}"
    )
    print(f"  -> 분석 완료 ({len(analysis)} 글자)")
    print(f"  -> 미리보기: {analysis[:150]}...\n")
    
    # Step 3: 실행 요약 (Step 2의 출력이 입력)
    print("Step 3: 실행 가능한 요약 생성 중...")
    summary = chain_step(
        system_prompt=(
            "You are a project manager. Create an actionable summary "
            "with prioritized action items based on the analysis. "
            "Respond in Korean."
        ),
        user_message=f"Create actionable summary from:\n\n{analysis}"
    )
    print(f"  -> 요약 완료 ({len(summary)} 글자)")
    
    return {
        "extracted": extracted,
        "analysis": analysis,
        "summary": summary
    }

results = document_analysis_chain(sample_document)
print("\n" + "=" * 50)
print("📊 최종 요약")
print("=" * 50)
print(results["summary"])

## §3. Gate 패턴 — 중간 품질 검증

다음 단계로 진행하기 전에 **품질 검증 체크포인트**를 삽입합니다.  
점수가 기준 미달이면 재시도합니다.

In [ ]:
def chain_with_gate(document_text):
    """Gate가 포함된 체이닝 파이프라인"""
    
    # Step 1: 핵심 정보 추출
    print("Step 1: 핵심 정보 추출...")
    extracted = chain_step(
        system_prompt="Extract key technical requirements from the document. "
                      "Be thorough and precise. Respond in Korean.",
        user_message=document_text
    )
    
    # Gate: 추출 품질 검증
    print("Gate: 추출 품질 검증...")
    quality_check = chain_step(
        system_prompt=(
            "You are a QA checker. Evaluate if the extraction is complete "
            "and accurate. Score 1-10. "
            'Respond ONLY with JSON: {"score": N, "issues": ["issue1", ...]}'
        ),
        user_message=f"Original:\n{document_text}\n\nExtraction:\n{extracted}"
    )
    
    try:
        check = json.loads(quality_check)
        score = check.get("score", 0)
    except json.JSONDecodeError:
        print("  ⚠️ JSON 파싱 실패, 기본 통과 처리")
        score = 7
    
    if score >= 7:
        print(f"  ✅ Gate 통과 (score={score}/10)")
    else:
        print(f"  ❌ Gate 실패 (score={score}/10), 재추출...")
        extracted = chain_step(
            system_prompt="Extract key requirements more thoroughly. "
                          "Address these issues: " + str(check.get("issues", [])),
            user_message=document_text
        )
        print("  🔄 재추출 완료")
    
    # Step 2: 분석 (Gate 통과 후)
    print("Step 2: 분석 진행...")
    analysis = chain_step(
        system_prompt="Analyze the extracted requirements and identify risks. "
                      "Respond in Korean.",
        user_message=extracted
    )
    
    return {"extracted": extracted, "analysis": analysis}

gate_results = chain_with_gate(sample_document)
print("\n📊 분석 결과")
print(gate_results["analysis"])

## §4. 핵심 정리

- **체이닝 핵심**: 이전 단계 출력 → 다음 단계 입력으로 연결
- **Gate 패턴**: 다음 단계 진행 전 품질 검증 체크포인트
- **단일 책임**: 각 단계는 하나의 작업에만 집중
- **다음 노트북**: `S8_04_routing.ipynb`에서 라우팅 워크플로를 구현한다